# Experiment 1: Baseline: Sentence-Level ModernBERT

This notebook implements the sentence-level ModernBERT baseline for CoNLL-2003 named entity recognition.

Our goal is to:

1. Load the original CoNLL-2003 formatted files
2. Parse them into sentence-level examples
3. Convert the data into Hugging Face `Dataset` format
4. Tokenize each sentence with `answerdotai/ModernBERT-base`
5. Align word-level NER labels to tokenized subwords
6. Fine-tune a standard token classification head
7. Evaluate entity-level F1 using `seqeval`

This notebook corresponds to the sentence-level ModernBERT baseline in our ablation framework.

## Imports and project paths
We first import the packages needed for preprocessing, tokenization, modeling, training, and evaluation.

We also define the project root and the location of the original CoNLL-2003 raw files.

In [2]:
from pathlib import Path
import os
import numpy as np
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

Was having issues with local paths so incldued code below

In [4]:
from pathlib import Path
import os

def find_repo_root(start: Path = Path.cwd()):
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            return p
    raise RuntimeError("Repo root not found")

PROJECT_ROOT = find_repo_root()
os.chdir(PROJECT_ROOT)        # optional: set CWD so other relative code works
conll_dir = PROJECT_ROOT / "data" / "conll2003"

## 1) Load and parse CoNLL-2003 files
Load the raw CoNLL files and convert to sentence-level examples with word-level NER tags.

In [5]:
def read_conll_sentences(path):
    from pathlib import Path as _Path
    path = _Path(path)
    sentences = []
    cur_words = []
    cur_labels = []

    with open(str(path), encoding="utf-8") as f:
        print(f"Reading: {path} (exists={path.exists()})")
        for line in f:
            line = line.strip()

            # Skip document boundary markers for the sentence-level baseline
            if line.startswith("-DOCSTART-"):
                continue

            if not line:
                if cur_words:
                    sentences.append({"words": cur_words, "ner_tags": cur_labels})
                    cur_words = []
                    cur_labels = []
                continue

            # CoNLL columns: word POS CHUNK NER
            parts = line.split()
            word = parts[0]
            ner = parts[-1] if len(parts) >= 4 else "O"

            cur_words.append(word)
            cur_labels.append(ner)

    if cur_words:
        sentences.append({"words": cur_words, "ner_tags": cur_labels})

    return sentences

# Locate files in project
conll_dir = Path("data/conll2003")
train_path = conll_dir / "eng.train"
valid_path = conll_dir / "eng.testa"
test_path = conll_dir / "eng.testb"

train_sents = read_conll_sentences(train_path)
valid_sents = read_conll_sentences(valid_path)
test_sents = read_conll_sentences(test_path)

len(train_sents), len(valid_sents), len(test_sents)


Reading: data/conll2003/eng.train (exists=True)
Reading: data/conll2003/eng.testa (exists=True)
Reading: data/conll2003/eng.testb (exists=True)


(14041, 3250, 3453)

## 2) Convert to Hugging Face `Dataset` format

In [6]:
def to_hf_format(sentences):
    examples = []
    for s in sentences:
        examples.append({'tokens': s['words'], 'ner_tags': s['ner_tags']})
    return examples

train_ds = Dataset.from_list(to_hf_format(train_sents))
valid_ds = Dataset.from_list(to_hf_format(valid_sents))
test_ds = Dataset.from_list(to_hf_format(test_sents))

dataset = DatasetDict({'train': train_ds, 'validation': valid_ds, 'test': test_ds})
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3453
    })
})

## 3) Tokenize and align labels to tokenized subwords
We use the fast tokenizer to align word-level labels to subword tokens. Subword tokens that are not the first token of a word get the label -100 so the loss ignores them.

In [7]:
model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Fixed CoNLL-2003 label mapping
label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG",
              "B-LOC", "I-LOC", "B-MISC", "I-MISC"]

label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for i, label in enumerate(label_list)}

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    new_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev_word = None

        for word_idx in word_ids:
            if word_idx is None:
                aligned.append(-100)
            elif word_idx != prev_word:
                aligned.append(label_to_id[labels[word_idx]])
            else:
                aligned.append(-100)
            prev_word = word_idx

        new_labels.append(aligned)

    tokenized["labels"] = new_labels
    return tokenized

tokenized_datasets = dataset.map(
    tokenize_and_align,
    batched=True,
    remove_columns=["tokens", "ner_tags"]
)

list(tokenized_datasets["train"].features.keys())


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

['input_ids', 'attention_mask', 'labels']

In [8]:
# 4) Model, trainer, and training arguments

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model_name = "answerdotai/ModernBERT-base"
num_labels = len(label_list)

# Standard token classification head with CoNLL-2003 labels
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id_to_label,
    label2id=label_to_id
)

data_collator = DataCollatorForTokenClassification(tokenizer)

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(preds, labels):
        cur_preds = []
        cur_labels = []

        for p_i, l_i in zip(pred, lab):
            if l_i == -100:
                continue
            cur_preds.append(id_to_label[int(p_i)])
            cur_labels.append(id_to_label[int(l_i)])

        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

training_args = TrainingArguments(
    output_dir="./modernbert-tokenclf",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


CUDA available: True
GPU: NVIDIA GeForce RTX 3090


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## 5) Train and evaluate
Run `trainer.train()` to fine-tune and then evaluate on the test set.

In [9]:
#run training
trainer.train()

#evaluate after training
val_metrics = trainer.evaluate(tokenized_datasets["validation"])
test_metrics = trainer.evaluate(tokenized_datasets["test"])
print("Validation metrics:", val_metrics)
print("Test metrics:", test_metrics)

print("Notebook ready: preprocessing, tokenization, trainer, and evaluation are configured.")
print("If CUDA available is True above, training will use a GPU-backed session.")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.119544,0.061080,0.891618,0.904073,0.897802,0.983743
2,0.028643,0.044793,0.930994,0.937731,0.934351,0.989214
3,0.009604,0.048631,0.943434,0.943117,0.943276,0.989662
4,0.003001,0.051743,0.928062,0.942275,0.935115,0.989097
5,0.000883,0.054872,0.935887,0.945809,0.940822,0.989642


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: on_train_begin must be called before on_evaluate